# Multi-Head Attention From scratch


In [1]:
import torch

inputs = torch.tensor(
    [[0.72, 0.45, 0.31],  # Dream   (x^1)
     [0.75, 0.20, 0.55],  # big     (x^2)
     [0.30, 0.80, 0.40],  # and     (x^3)
     [0.85, 0.35, 0.60],  # work    (x^4)
     [0.55, 0.15, 0.75],  # for     (x^5)
     [0.25, 0.20, 0.85]]  # it      (x^6)
)

# Corresponding words
words = ['Dream', 'big', 'and', 'work', 'for', 'it']


In [2]:
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.d_out = d_out
        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # Causal mask (upper triangular, diagonal=1)
        self.register_buffer("mask",torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        # x shape: (b, num_tokens, d_in)
        b, num_tokens, d_in = x.shape  # New batch dimension b
        keys    = self.w_key(x)
        queries = self.w_query(x)
        values  = self.w_value(x)

        # Attention scores
        attn_scores = queries @ keys.transpose(1, 2)  # Changed transpose

        # Apply causal mask
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens],-torch.inf)

        # Softmax
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5,dim=1)
        attn_weights = self.dropout(attn_weights)  # New

        context_vec = attn_weights @ values

        return context_vec


In [3]:
d_in=inputs.shape[-1]
d_out=2
print(d_in,d_out)

3 2


In [4]:
batch=torch.stack((inputs,inputs),dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [5]:
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
    super().__init__()

    self.heads=nn.ModuleList(
        [CausalAttention(d_in,d_out,context_length,dropout,qkv_bias)
        for _ in range(num_heads)]
    )

  def forward(self,x):
    return torch.cat([head(x) for head in self.heads],dim=-1 )

In [6]:
torch.manual_seed(123)
context_length=batch.shape[1] # this is the no of tokens is 6
d_in , d_out = 3 , 2
mha=MultiHeadAttentionWrapper(d_in,d_out,context_length,dropout=0.0,num_heads=2)

In [7]:
context_vec = mha(batch)
print(context_vec)
print("Context vector shape = :",context_vec.shape)

tensor([[[-0.0928, -0.0262,  0.0916,  0.0598],
         [-0.2124, -0.0207,  0.2075,  0.1080],
         [-0.3058, -0.0742,  0.3175,  0.2096],
         [-0.5702, -0.0896,  0.5508,  0.3245],
         [-0.8150, -0.0104,  0.8001,  0.3963],
         [-1.1663,  0.1943,  1.1814,  0.4830]],

        [[-0.0928, -0.0262,  0.0916,  0.0598],
         [-0.2124, -0.0207,  0.2075,  0.1080],
         [-0.3058, -0.0742,  0.3175,  0.2096],
         [-0.5702, -0.0896,  0.5508,  0.3245],
         [-0.8150, -0.0104,  0.8001,  0.3963],
         [-1.1663,  0.1943,  1.1814,  0.4830]]], grad_fn=<CatBackward0>)
Context vector shape = : torch.Size([2, 6, 4])
